In [1]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
import lightgbm as lgb
import joblib
from sklearn.model_selection import GroupKFold
from src.data import load_clean
from src.features import add_distance_features, add_listing_features

df = add_listing_features(add_distance_features(load_clean()))

num_cols = ["accommodates", "bedrooms", "beds", "latitude", "longitude",
            "dist_colosseum_km", "dist_termini_km", "dist_st_peters_km",
            "dist_pantheon_km", "dist_trevi_km", "dist_nearest_landmark_km",
            "n_amenities", "bathrooms_num", "minimum_nights"]
cat_cols = ["room_type", "neighbourhood_cleansed", "property_type"]

X = df[num_cols + cat_cols].copy()
for c in cat_cols:
    X[c] = X[c].astype("category")
y = df["log_price"]
print(X.shape)  # ожидаем (33769, 17)

(33769, 17)


In [2]:
def make_model(alpha=None):
    params = dict(n_estimators=500, learning_rate=0.05,
                  num_leaves=31, random_state=42, verbose=-1)
    if alpha is not None:
        params.update(objective="quantile", alpha=alpha)
    return lgb.LGBMRegressor(**params)

In [3]:
lo = np.zeros(len(df)); hi = np.zeros(len(df))
for tr, te in GroupKFold(n_splits=5).split(X, y, df["host_id"]):
    m_lo = make_model(0.2).fit(X.iloc[tr], y.iloc[tr])
    m_hi = make_model(0.8).fit(X.iloc[tr], y.iloc[tr])
    lo[te] = m_lo.predict(X.iloc[te])
    hi[te] = m_hi.predict(X.iloc[te])

lo, hi = np.minimum(lo, hi), np.maximum(lo, hi)   # на случай пересечения
inside = (y.values >= lo) & (y.values <= hi)
print("Coverage of the 20-80% range:", round(inside.mean() * 100, 1), "% (target ~60%)")
print("Median range width (price):",
      round(np.median(np.expm1(hi) - np.expm1(lo)), 1))

Coverage of the 20-80% range: 54.8 % (target ~60%)
Median range width (price): 81.2


In [4]:
final = {
    "point": make_model().fit(X, y),
    "q_low": make_model(0.2).fit(X, y),
    "q_high": make_model(0.8).fit(X, y),
    "features": list(X.columns),
    "num_cols": num_cols,
    "cat_cols": cat_cols,
    "categories": {c: list(X[c].cat.categories) for c in cat_cols},
}
joblib.dump(final, "../models/price_model.joblib", compress=3)

import os
print(round(os.path.getsize("../models/price_model.joblib") / 1e6, 1), "MB")

1.6 MB


In [5]:
loaded = joblib.load("../models/price_model.joblib")
sample = X.iloc[[0]]
print(np.expm1(loaded["q_low"].predict(sample)),
      np.expm1(loaded["point"].predict(sample)),
      np.expm1(loaded["q_high"].predict(sample)),
      "actual:", df["price"].iloc[0])

[75.074208] [107.22227577] [145.29313636] actual: 66.0


In [6]:
df[["latitude", "longitude", "neighbourhood_cleansed"]].to_csv(
    "../models/neighbourhood_lookup.csv", index=False)

In [7]:
pip install streamlit

   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
    --------------------------------------- 0.3/11.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.2 MB 1.8 MB/s eta 0:00:07
   --- ------------------------------------ 1.0/11.2 MB 2.2 MB/s eta 0:00:05
   ------ --------------------------------- 1.8/11.2 MB 2.5 MB/s eta 0:00:04
   --------- ------------------------------ 2.6/11.2 MB 2.7 MB/s eta 0:00:04
   ------------- -------------------------- 3.7/11.2 MB 3.1 MB/s eta 0:00:03
   ------------------ --------------------- 5.2/11.2 MB 3.7 MB/s eta 0:00:02
   ------------------------ --------------- 6.8/11.2 MB 4.2 MB/s eta 0:00:02
   ---------------------------- ----------- 8.1/11.2 MB 4.5 MB/s eta 0:00:01
   ------------------------------- -------- 8.9/11.2 MB 4.4 MB/s eta 0:00:01
   ------------------------------------ --- 10.2/11.2 MB 4.5 MB/s eta 0:00:01
   ---------------------------------------  11.0/11.2 MB 4.6 MB/s eta 0:00:01
   --------


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
